# Optimización de hiperparámetros en aprendizaje no supervisado

En este notebook vamos a construir un flujo completo de **clustering no supervisado** y comparar cuatro estrategias de optimización de hiperparámetros: **Grid Search**, **Random Search**, **optimización bayesiana** y una estrategia **basada en gradientes**. La idea es aplicar estas técnicas sobre un problema real de agrupamiento de textos extraídos desde Hugging Face.

El modelo elegido será **K-Means**, un algoritmo de clustering ampliamente utilizado en scikit-learn [web:1][web:2]. Para evaluar la calidad de los clusters usaremos la métrica **silhouette score**, que mide simultáneamente cohesión intra-cluster y separación entre clusters [web:19][web:22]. Además, usaremos **BayesSearchCV** de scikit-optimize como aproximación de búsqueda bayesiana de hiperparámetros [web:23].

El dataset utilizado es `Roman1111111/claude-opus-4.6-10000x`, publicado en Hugging Face, descrito como un dataset de razonamiento de alta fidelidad sintetizado con Claude Opus 4.6 [web:7][web:17].

In [ ]:
# Instalamos las librerías necesarias en Google Colab
# scikit-optimize se utilizará para la optimización bayesiana
!pip -q install scikit-optimize datasets

## 1. Carga del dataset

En esta sección cargamos el dataset directamente desde su ruta en Hugging Face usando `pandas.read_json`. Después inspeccionaremos su estructura para identificar qué columnas nos interesan y transformar el contenido textual en una representación numérica apta para clustering.

Como el dataset está orientado a razonamiento y conversaciones, una forma práctica de trabajar es convertir cada fila en un texto consolidado. Así podremos vectorizarlo con **TF-IDF**, una técnica clásica para representar documentos en forma matricial numérica.

In [ ]:
# Importamos librerías base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargamos el dataset desde Hugging Face
df = pd.read_json(
    'hf://datasets/Roman1111111/claude-opus-4.6-10000x/opus46_final.jsonl',
    lines=True
)

# Mostramos tamaño y primeras filas
print('Forma del dataset:', df.shape)
display(df.head())

# Mostramos nombres de columnas
print('Columnas:', df.columns.tolist())

## 2. Preparación del texto

Como el modelo será no supervisado, no necesitamos una variable objetivo. Lo importante es convertir cada observación en un vector numérico que conserve suficiente información semántica para que el algoritmo pueda detectar grupos.

Vamos a crear una columna `text` a partir de la estructura de cada fila. El código está preparado para adaptarse si el dataset contiene listas de mensajes, diccionarios o texto ya consolidado. Después tomaremos una muestra para que la ejecución en Colab sea razonable.

In [ ]:
# Función auxiliar para convertir cada fila en un texto único
def row_to_text(row):
    parts = []
    
    # Recorremos todas las columnas para construir una representación textual robusta
    for col in row.index:
        value = row[col]
        
        # Si el valor es una lista, intentamos concatenar su contenido
        if isinstance(value, list):
            for item in value:
                if isinstance(item, dict):
                    parts.append(' '.join([str(v) for v in item.values()]))
                else:
                    parts.append(str(item))
        
        # Si el valor es un diccionario, concatenamos sus valores
        elif isinstance(value, dict):
            parts.append(' '.join([str(v) for v in value.values()]))
        
        # En otro caso, lo convertimos directamente en texto
        elif pd.notnull(value):
            parts.append(str(value))
    
    # Devolvemos un único string limpio
    return ' '.join(parts).strip()

# Creamos una columna textual
df['text'] = df.apply(row_to_text, axis=1)

# Eliminamos textos vacíos o demasiado cortos
df = df[df['text'].str.len() > 30].copy()

# Para que el notebook sea manejable en Colab, usamos una muestra
sample_size = min(2500, len(df))
df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)

print('Número de textos usados:', len(df_sample))
display(df_sample[['text']].head())

## 3. Vectorización TF-IDF

K-Means trabaja con variables numéricas, así que debemos transformar los textos. Para ello usamos **TF-IDF**, limitando el vocabulario y eliminando palabras demasiado frecuentes o demasiado raras para mejorar estabilidad y velocidad.

Después reduciremos dimensionalidad con **TruncatedSVD**. Esto ayuda a que K-Means trabaje mejor en espacios textuales dispersos y también hace que la optimización sea más rápida.

In [ ]:
# Importamos herramientas de preprocesado
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline

# Definimos una pipeline de representación textual
text_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=3000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9
    )),
    ('svd', TruncatedSVD(n_components=100, random_state=42)),
    ('norm', Normalizer())
])

# Transformamos el texto a una matriz numérica densa y reducida
X = text_pipeline.fit_transform(df_sample['text'])

print('Forma de la matriz final:', X.shape)

## 4. Función objetivo para la optimización

Como estamos ante un problema no supervisado, no podemos usar `accuracy` ni métricas de clasificación. En su lugar usaremos el **silhouette score**, que es una métrica estándar para valorar clusters generados por K-Means [web:19][web:22].

Nuestra función objetivo recibirá hiperparámetros, entrenará K-Means y devolverá la puntuación de silhouette. De esta manera, cualquier estrategia de optimización podrá evaluar distintas combinaciones de parámetros sobre la misma base.

In [ ]:
# Importamos K-Means y la métrica de evaluación
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import time

# Función que entrena K-Means y devuelve la silhouette
def evaluate_kmeans(n_clusters, init, n_init, max_iter, algorithm):
    # Convertimos los parámetros discretos a tipos adecuados
    n_clusters = int(n_clusters)
    n_init = int(n_init)
    max_iter = int(max_iter)
    
    # K-Means necesita al menos 2 clusters
    if n_clusters < 2:
        return -1
    
    # Entrenamos el modelo
    model = KMeans(
        n_clusters=n_clusters,
        init=init,
        n_init=n_init,
        max_iter=max_iter,
        algorithm=algorithm,
        random_state=42
    )
    
    labels = model.fit_predict(X)
    
    # Si por alguna razón solo aparece un cluster efectivo, devolvemos mala puntuación
    if len(np.unique(labels)) < 2:
        return -1
    
    # Calculamos la silhouette
    score = silhouette_score(X, labels)
    return score

## 5. Espacio de búsqueda

Vamos a optimizar varios hiperparámetros de K-Means. Los más relevantes aquí son el número de clusters `n_clusters`, el método de inicialización `init`, el número de reinicios `n_init`, el máximo de iteraciones `max_iter` y el algoritmo interno usado por scikit-learn.

La documentación de scikit-learn muestra que K-Means permite ajustar este tipo de parámetros y que la selección del número de clusters es especialmente importante en la práctica [web:1][web:2][web:19].

In [ ]:
# Definimos el espacio de búsqueda común para las estrategias
param_space = {
    'n_clusters': [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15],
    'init': ['k-means++', 'random'],
    'n_init': [5, 10, 15, 20],
    'max_iter': [100, 200, 300, 500],
    'algorithm': ['lloyd', 'elkan']
}

## 6. Grid Search manual

La búsqueda en rejilla o **Grid Search** evalúa sistemáticamente combinaciones predefinidas de hiperparámetros. Su ventaja es que explora el espacio de manera exhaustiva dentro de una rejilla concreta, aunque puede ser costosa cuando el número de combinaciones crece.

Aquí la implementamos de forma manual porque estamos en un contexto no supervisado y queremos controlar la métrica directamente.

In [ ]:
# Importamos utilidades para generar todas las combinaciones
from itertools import product

# Lista donde guardaremos resultados
grid_results = []

# Tomamos el tiempo de inicio
start_time = time.time()

# Generamos todas las combinaciones posibles del grid
all_combinations = list(product(
    param_space['n_clusters'],
    param_space['init'],
    param_space['n_init'],
    param_space['max_iter'],
    param_space['algorithm']
))

# Recorremos cada combinación y evaluamos el modelo
for n_clusters, init, n_init, max_iter, algorithm in all_combinations:
    score = evaluate_kmeans(n_clusters, init, n_init, max_iter, algorithm)
    grid_results.append({
        'method': 'Grid Search',
        'n_clusters': n_clusters,
        'init': init,
        'n_init': n_init,
        'max_iter': max_iter,
        'algorithm': algorithm,
        'score': score
    })

# Calculamos tiempo total
grid_time = time.time() - start_time

# Convertimos a DataFrame y mostramos mejores resultados
grid_results_df = pd.DataFrame(grid_results).sort_values('score', ascending=False)
print(f'Tiempo Grid Search: {grid_time:.2f} segundos')
display(grid_results_df.head())

## 7. Random Search manual

La **Random Search** no recorre exhaustivamente todas las combinaciones, sino que prueba configuraciones elegidas al azar. Esto suele ser más eficiente cuando no todos los hiperparámetros influyen por igual en el resultado, una idea muy conocida en optimización de hiperparámetros [web:9][web:12].

En este caso fijaremos un número de iteraciones para tener una comparación justa frente al coste de Grid Search.

In [ ]:
# Fijamos semilla para reproducibilidad
rng = np.random.default_rng(42)

# Número de configuraciones aleatorias a evaluar
n_random_trials = 40

# Lista de resultados
random_results = []

# Tiempo inicial
start_time = time.time()

# Ejecutamos distintas pruebas aleatorias
for _ in range(n_random_trials):
    n_clusters = rng.choice(param_space['n_clusters'])
    init = rng.choice(param_space['init'])
    n_init = rng.choice(param_space['n_init'])
    max_iter = rng.choice(param_space['max_iter'])
    algorithm = rng.choice(param_space['algorithm'])
    
    score = evaluate_kmeans(n_clusters, init, n_init, max_iter, algorithm)
    random_results.append({
        'method': 'Random Search',
        'n_clusters': int(n_clusters),
        'init': str(init),
        'n_init': int(n_init),
        'max_iter': int(max_iter),
        'algorithm': str(algorithm),
        'score': score
    })

# Tiempo total
random_time = time.time() - start_time

# Convertimos a DataFrame
random_results_df = pd.DataFrame(random_results).sort_values('score', ascending=False)
print(f'Tiempo Random Search: {random_time:.2f} segundos')
display(random_results_df.head())

## 8. Optimización bayesiana

La **optimización bayesiana** intenta aprender qué regiones del espacio de búsqueda son prometedoras a medida que avanza la exploración. En lugar de probar puntos al azar o de forma exhaustiva, utiliza la información de evaluaciones anteriores para decidir dónde buscar después.

Usaremos `BayesSearchCV` de scikit-optimize, una herramienta diseñada precisamente para búsqueda bayesiana de hiperparámetros [web:23]. Como estamos en un problema no supervisado, definiremos un estimador compatible con scikit-learn y una métrica personalizada basada en silhouette.

In [ ]:
# Importamos clases necesarias para integrar la búsqueda bayesiana
from sklearn.base import BaseEstimator, ClusterMixin
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# Creamos un wrapper compatible con scikit-learn
class KMeansWrapper(BaseEstimator, ClusterMixin):
    def __init__(self, n_clusters=8, init='k-means++', n_init=10, max_iter=300, algorithm='lloyd'):
        self.n_clusters = n_clusters
        self.init = init
        self.n_init = n_init
        self.max_iter = max_iter
        self.algorithm = algorithm
    
    def fit(self, X, y=None):
        # Entrenamos el modelo K-Means interno
        self.model_ = KMeans(
            n_clusters=self.n_clusters,
            init=self.init,
            n_init=self.n_init,
            max_iter=self.max_iter,
            algorithm=self.algorithm,
            random_state=42
        )
        self.labels_ = self.model_.fit_predict(X)
        return self
    
    def predict(self, X):
        # Devolvemos la asignación de cluster
        return self.model_.predict(X)

# Definimos una función de scoring no supervisada
def unsupervised_scorer(estimator, X, y=None):
    estimator.fit(X)
    labels = estimator.labels_
    if len(np.unique(labels)) < 2:
        return -1
    return silhouette_score(X, labels)

# Espacio de búsqueda bayesiano
search_spaces = {
    'n_clusters': Integer(2, 15),
    'init': Categorical(['k-means++', 'random']),
    'n_init': Integer(5, 20),
    'max_iter': Integer(100, 500),
    'algorithm': Categorical(['lloyd', 'elkan'])
}

# Instanciamos la búsqueda bayesiana
bayes_search = BayesSearchCV(
    estimator=KMeansWrapper(),
    search_spaces=search_spaces,
    n_iter=30,
    scoring=unsupervised_scorer,
    cv=[(slice(None), slice(None))],
    n_jobs=-1,
    random_state=42,
    refit=False
)

# Ejecutamos la búsqueda bayesiana
start_time = time.time()
bayes_search.fit(X)
bayes_time = time.time() - start_time

# Extraemos resultados
bayes_results_df = pd.DataFrame(bayes_search.cv_results_).sort_values('mean_test_score', ascending=False)
print(f'Tiempo Bayesiano: {bayes_time:.2f} segundos')
display(bayes_results_df[['params', 'mean_test_score']].head())

## 9. Optimización basada en gradientes

En modelos clásicos como K-Means, los hiperparámetros no siempre son diferenciables directamente. Por eso, una optimización basada en gradientes no se aplica de la misma manera que en redes neuronales.

Sin embargo, sí podemos construir una estrategia inspirada en gradientes sobre un **parámetro continuo relajado**, aproximando el máximo local de la función objetivo. Para este ejercicio usaremos una versión simple con diferencias finitas sobre el número de clusters tratado como variable continua y redondeado al evaluar. No es una optimización por gradiente exacta del algoritmo interno, pero sí una búsqueda basada en la idea de derivada numérica sobre la función objetivo.

In [ ]:
# Función objetivo simplificada para la búsqueda basada en gradientes
# Aquí fijamos algunos hiperparámetros y optimizamos principalmente n_clusters
def objective_relaxed(k_continuous):
    # Convertimos el valor continuo al entero válido más cercano
    k = int(np.clip(np.round(k_continuous), 2, 15))
    return evaluate_kmeans(k, 'k-means++', 10, 300, 'lloyd')

# Implementamos ascenso por gradiente numérico con diferencias finitas
def gradient_based_search(k0=6.0, lr=0.8, h=1.0, max_steps=12):
    history = []
    k = float(k0)
    
    for step in range(max_steps):
        # Evaluamos la función objetivo en el punto actual
        score_current = objective_relaxed(k)
        
        # Aproximamos la derivada mediante diferencias finitas centradas
        score_plus = objective_relaxed(k + h)
        score_minus = objective_relaxed(k - h)
        grad = (score_plus - score_minus) / (2 * h)
        
        # Actualizamos el parámetro moviéndonos en dirección ascendente
        k = k + lr * grad * 10
        k = float(np.clip(k, 2, 15))
        
        history.append({
            'step': step,
            'k_continuous': k,
            'k_integer': int(np.round(k)),
            'score': score_current,
            'gradient': grad
        })
    
    return pd.DataFrame(history)

# Ejecutamos la búsqueda basada en gradientes
start_time = time.time()
grad_results_df = gradient_based_search()
grad_time = time.time() - start_time

print(f'Tiempo Gradiente: {grad_time:.2f} segundos')
display(grad_results_df)

## 10. Comparación de resultados

Ahora reunimos los mejores resultados de las cuatro estrategias. El objetivo es comparar tanto la **mejor silhouette obtenida** como el **tiempo de ejecución**.

Esto nos permitirá discutir qué método fue más exhaustivo, cuál fue más eficiente y cuál encontró la mejor solución para este problema concreto.

In [ ]:
# Extraemos el mejor resultado de Grid Search
best_grid = grid_results_df.iloc[0]

# Extraemos el mejor resultado de Random Search
best_random = random_results_df.iloc[0]

# Extraemos el mejor resultado de Bayes Search
best_bayes = bayes_results_df.iloc[0]

# Extraemos el mejor resultado del método basado en gradientes
best_grad = grad_results_df.sort_values('score', ascending=False).iloc[0]

# Construimos un resumen comparativo
comparison_df = pd.DataFrame([
    {
        'method': 'Grid Search',
        'best_score': best_grid['score'],
        'time_seconds': grid_time,
        'best_params': {
            'n_clusters': int(best_grid['n_clusters']),
            'init': best_grid['init'],
            'n_init': int(best_grid['n_init']),
            'max_iter': int(best_grid['max_iter']),
            'algorithm': best_grid['algorithm']
        }
    },
    {
        'method': 'Random Search',
        'best_score': best_random['score'],
        'time_seconds': random_time,
        'best_params': {
            'n_clusters': int(best_random['n_clusters']),
            'init': best_random['init'],
            'n_init': int(best_random['n_init']),
            'max_iter': int(best_random['max_iter']),
            'algorithm': best_random['algorithm']
        }
    },
    {
        'method': 'Bayesian Optimization',
        'best_score': best_bayes['mean_test_score'],
        'time_seconds': bayes_time,
        'best_params': best_bayes['params']
    },
    {
        'method': 'Gradient-Based Search',
        'best_score': best_grad['score'],
        'time_seconds': grad_time,
        'best_params': {
            'n_clusters': int(best_grad['k_integer']),
            'init': 'k-means++',
            'n_init': 10,
            'max_iter': 300,
            'algorithm': 'lloyd'
        }
    }
])

# Mostramos la tabla resumen
display(comparison_df.sort_values('best_score', ascending=False))

## 11. Visualización comparativa

Una gráfica simple ayuda a ver rápidamente qué estrategia obtuvo mejor puntuación y cuánto costó computacionalmente. En problemas reales, esta diferencia puede ser muy importante, porque una mejora pequeña en calidad puede no compensar un coste enorme de tiempo.

In [ ]:
# Configuramos estilo visual
sns.set(style='whitegrid')

# Creamos una figura con dos gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de mejor score por método
sns.barplot(data=comparison_df, x='method', y='best_score', ax=axes[0], palette='viridis')
axes[0].set_title('Mejor silhouette score por estrategia')
axes[0].set_xlabel('Método')
axes[0].set_ylabel('Silhouette score')
axes[0].tick_params(axis='x', rotation=20)

# Gráfico de tiempo por método
sns.barplot(data=comparison_df, x='method', y='time_seconds', ax=axes[1], palette='magma')
axes[1].set_title('Tiempo de ejecución por estrategia')
axes[1].set_xlabel('Método')
axes[1].set_ylabel('Segundos')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## 12. Entrenamiento final con la mejor configuración

Finalmente entrenamos un modelo K-Means usando la mejor configuración encontrada entre todos los métodos. Esto permite inspeccionar la distribución final de clusters y dejar listo el modelo para análisis posterior.

En un trabajo académico, aquí puedes añadir una interpretación de los grupos generados, revisar ejemplos de textos dentro de cada cluster y comentar qué estrategia te pareció más adecuada en términos de calidad y coste computacional.

In [ ]:
# Seleccionamos la mejor fila de la tabla comparativa
overall_best = comparison_df.sort_values('best_score', ascending=False).iloc[0]
print('Mejor método encontrado:', overall_best['method'])
print('Mejores parámetros:', overall_best['best_params'])

# Extraemos parámetros
best_params = overall_best['best_params']

# Entrenamos K-Means final con la mejor configuración
final_model = KMeans(
    n_clusters=int(best_params['n_clusters']),
    init=best_params['init'],
    n_init=int(best_params['n_init']),
    max_iter=int(best_params['max_iter']),
    algorithm=best_params['algorithm'],
    random_state=42
)

# Generamos etiquetas finales
df_sample['cluster'] = final_model.fit_predict(X)

# Mostramos distribución de clusters
print(df_sample['cluster'].value_counts().sort_index())

# Mostramos algunos ejemplos por cluster
for cluster_id in sorted(df_sample['cluster'].unique()):
    print(f'\n===== Cluster {cluster_id} =====')
    examples = df_sample[df_sample['cluster'] == cluster_id]['text'].head(2).tolist()
    for i, example in enumerate(examples, 1):
        print(f'Ejemplo {i}:', example[:300], '...')

## 13. Comentario final para la memoria o entrega

Este ejercicio muestra que las técnicas de optimización también pueden aplicarse a problemas no supervisados siempre que definamos una función objetivo adecuada, como la silhouette score [web:19][web:22]. Grid Search ofrece una exploración exhaustiva pero puede ser costosa; Random Search reduce el coste probando puntos al azar y suele ser competitiva [web:9][web:12]; la optimización bayesiana aprovecha mejor la información acumulada durante la búsqueda [web:23].

La estrategia basada en gradientes incluida aquí debe interpretarse como una aproximación didáctica sobre una función objetivo relajada, ya que los hiperparámetros discretos de K-Means no son naturalmente diferenciables. Aun así, sirve para comparar una idea de búsqueda guiada frente a métodos exhaustivos, aleatorios y bayesianos.